**Cell 1**

# 01 — Aggregate Per-Class Datasets to 200/class

For each class we've downloaded **one or more** Roboflow/Kaggle exports. Each export is single-class and shaped like:

```
datasets/<class>/<dataset_name>/
├── train/{images,labels}
├── valid/{images,labels}
├── test/{images,labels}
└── data.yaml
```

A class folder may contain one sub-dataset or several — the notebook walks whatever is there.

This notebook pools every image/label pair across *all* sub-datasets and *all* splits for a class, randomly samples **200**, and writes them into one flat per-class aggregated folder:

```
data/aggregated/<class>/
├── images/  (200 jpgs)
├── labels/  (200 YOLO txts, still class_id=0)
└── info.json   (provenance: which source each file came from)
```

Labels keep their original `class_id = 0` — the remap to the global id happens in notebook 02.

In [1]:
# Cell 2 — install dependencies for this notebook
%pip install -q pandas pillow pyyaml


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Cell 3 — setup
import json, random
from pathlib import Path
import pandas as pd
import yaml
from PIL import Image

random.seed(42)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']
TARGET_PER_CLASS = 200
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
SPLIT_DIRS = {'train', 'valid', 'val', 'test'}  # Roboflow uses 'valid'

# Raw sub-datasets live at <project_root>/datasets/<class>/<dataset_N>/
SRC = Path('../datasets')
DST = Path('../data/aggregated')
assert SRC.exists(), f'create per-class folders under {SRC.resolve()} and drop Roboflow/Kaggle exports into them'
for c in CLASSES:
    (DST / c / 'images').mkdir(parents=True, exist_ok=True)
    (DST / c / 'labels').mkdir(parents=True, exist_ok=True)

**Cell 4**

## 1.1 Discover every (image, label) pair across sub-datasets and splits
For each `datasets/<class>/` folder we walk into every sub-dataset (one or more) and collect pairs from `train/`, `valid/`, and `test/`. The `split` and source-dataset name are recorded for provenance.

In [2]:
# Cell 5 — discover pairs per class
def find_label(img_path: Path) -> Path | None:
    # Roboflow/Kaggle convention: sibling 'labels/<stem>.txt' to images/
    for cand in [img_path.parent.parent / 'labels' / (img_path.stem + '.txt'),
                 img_path.with_suffix('.txt')]:
        if cand.exists():
            return cand
    return None

def infer_split(img_path: Path) -> str:
    for part in img_path.parts[::-1]:
        if part in SPLIT_DIRS:
            return 'val' if part == 'valid' else part
    return 'train'

pairs_per_class: dict[str, list[dict]] = {}
for c in CLASSES:
    class_root = SRC / c
    if not class_root.exists():
        print(f'[missing] {class_root}'); pairs_per_class[c] = []; continue

    collected = []
    for sub in sorted(p for p in class_root.iterdir() if p.is_dir()):
        for img in sub.rglob('*'):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            lbl = find_label(img)
            if lbl is None:
                continue
            collected.append({
                'img': img, 'lbl': lbl,
                'source_dataset': sub.name,
                'source_split': infer_split(img),
            })
    pairs_per_class[c] = collected

pd.DataFrame([(c, len(p)) for c, p in pairs_per_class.items()],
             columns=['class', 'available_pairs'])

,class,available_pairs
0,projector,319
1,whiteboard,200
2,fire_extinguisher,848
3,door_sign,431


**Cell 6**

## 1.2 Per-class source breakdown
A quick view of where the available pairs are coming from (which sub-dataset + which split).

In [3]:
# Cell 7 — pairs per class per sub-dataset per split
rows = [{'class': c, **p} for c, pairs in pairs_per_class.items() for p in pairs]
breakdown = pd.DataFrame(rows).drop(columns=['img', 'lbl'], errors='ignore')
breakdown.groupby(['class', 'source_dataset', 'source_split']).size().unstack(fill_value=0)

source_split                         test  train  val
class             source_dataset                     
door_sign         door_signs_1          0    345   86
fire_extinguisher Fire Extinguisher    36    750   62
projector         Projector2            6     39   11
                  Projector3            1    170   28
                  projector1            6     45   13
whiteboard        Whiteboard           20    140   40

**Cell 8**

## 1.3 Sample 200 per class and copy into the aggregated folder
Shuffles the pool, picks `TARGET_PER_CLASS`, converts each image to RGB JPG (strips EXIF), renames to `<class>_<nnnn>.jpg`. Labels keep their original `class_id = 0`.

In [4]:
# Cell 9 — sample, copy, and record provenance
for c in CLASSES:
    pool = pairs_per_class[c]
    if len(pool) < TARGET_PER_CLASS:
        print(f'[warn] {c}: only {len(pool)} pairs available (< {TARGET_PER_CLASS})')
    random.shuffle(pool)
    selected = pool[:TARGET_PER_CLASS]

    info = {'class': c, 'count': len(selected), 'items': []}
    for i, p in enumerate(selected, 1):
        stem = f'{c}_{i:04d}'
        dst_img = DST / c / 'images' / f'{stem}.jpg'
        dst_lbl = DST / c / 'labels' / f'{stem}.txt'

        with Image.open(p['img']) as im:
            im.convert('RGB').save(dst_img, 'JPEG', quality=92)
        dst_lbl.write_text(p['lbl'].read_text())

        info['items'].append({
            'filename': f'{stem}.jpg',
            'source_dataset': p['source_dataset'],
            'source_split': p['source_split'],
            'original_image': str(p['img']),
        })

    (DST / c / 'info.json').write_text(json.dumps(info, indent=2))
    print(f'{c}: wrote {len(selected)} pairs to {DST / c}')

projector: wrote 200 pairs to ../data/aggregated/projector
whiteboard: wrote 200 pairs to ../data/aggregated/whiteboard
fire_extinguisher: wrote 200 pairs to ../data/aggregated/fire_extinguisher
door_sign: wrote 200 pairs to ../data/aggregated/door_sign


**Cell 10**

## 1.4 Verify the aggregated folders

In [5]:
# Cell 11 — confirm each class has 200 images + 200 labels + info.json
check = []
for c in CLASSES:
    imgs = list((DST / c / 'images').glob('*.jpg'))
    lbls = list((DST / c / 'labels').glob('*.txt'))
    info_ok = (DST / c / 'info.json').exists()
    check.append({'class': c, 'images': len(imgs), 'labels': len(lbls), 'info.json': info_ok})
pd.DataFrame(check)

,class,images,labels,info.json
0,projector,200,200,True
1,whiteboard,200,200,True
2,fire_extinguisher,200,200,True
3,door_sign,200,200,True


**Cell 12**

### Acceptance criteria
- Cell 11 shows 200 images + 200 labels for every class
- Cell 7's breakdown shows expected coverage across your sub-datasets
- Each `data/aggregated/<class>/info.json` records the provenance of every selected file

Proceed to **notebook 02** to combine the four aggregated folders into one training-ready split dataset.